# Шаблон LightningModule для проекта

Скопируйте этот файл под свою модель и замените места, помеченные `# TODO`:

1. **`MyDataModule`** — `prepare_data()`/`setup()`: как загружается ваш датасет, и трансформы
2. **`raw_model`** внутри `MyClassifier.__init__` — ваша архитектура (готовая через `timm` или свой класс)
3. **`mean`/`std`** — статистика нормализации именно вашего датасета
4. **Гиперпараметры** при создании `model = MyClassifier(...)` — `num_classes`, `lr`, `max_epochs` и т.д.

Всё остальное (структура `training_step`/`validation_step`, чекпоинты, сохранение весов) менять не нужно — оно одинаковое для любой модели в проекте.

In [ ]:
!pip install lightning -q
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint

SEED = 0
L.seed_everything(SEED)

# Загрузка Датасета, Даталоадеры

In [ ]:
class MyDataModule(L.LightningDataModule):
    """Скопируйте и переименуйте под свой датасет."""

    def __init__(self, data_dir="./data", batch_size=128, val_size=5000, num_workers=2, seed=SEED):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.val_size = val_size
        self.num_workers = num_workers
        self.seed = seed

        # TODO: замените на трансформы под свой датасет.
        # Без Normalize здесь — она должна быть ВНУТРИ модели (см. класс ниже),
        # чтобы атаки из src/attacks/ работали в чистом пространстве [0, 1].
        self.transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])
        self.transform_test = transforms.Compose([
            transforms.ToTensor(),
        ])

    def prepare_data(self):
        # TODO: скачивание датасета. Если запускаете на Kaggle — сначала проверьте
        # /kaggle/input на уже готовый датасет (пример поиска — в resnet18-ноутбуке проекта),
        # и только если не нашли — download=True.
        pass

    def setup(self, stage=None):
        # TODO: собрать self.train_set / self.val_set с нужными трансформами.
        # Пример разбиения на train/val через random_split — в resnet18-ноутбуке проекта.
        pass

    def train_dataloader(self):
        return DataLoader(
            self.train_set, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True,
        )


# TODO: параметры под ваш датасет
dm = MyDataModule(batch_size=128)

# Загрузка модели

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, -1, 1, 1))
        self.register_buffer("std",  torch.tensor(std).view(1, -1, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std


class NormalizedModel(nn.Module):
    """Оборачивает произвольную архитектуру, добавляя нормализацию на входе.
    Кладите сюда свою модель — тогда сохранённые веса сразу подходят под
    атаки из src/attacks/ без отдельной обёртки при загрузке."""
    def __init__(self, model, mean, std):
        super().__init__()
        self.normalize = Normalize(mean, std)
        self.model = model

    def forward(self, x):
        return self.model(self.normalize(x))


class MyClassifier(L.LightningModule):
    """Скопируйте и переименуйте под свою архитектуру.
    training_step/validation_step/configure_optimizers менять не нужно —
    они не зависят от конкретной модели."""

    def __init__(self, num_classes=10, lr=1e-3, weight_decay=0.05, max_epochs=100,
                 mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)):
        super().__init__()
        self.save_hyperparameters()

        # TODO: замените на свою архитектуру, например:
        #   raw_model = timm.create_model('...', pretrained=False, num_classes=num_classes)
        #   raw_model = MyCustomNet(num_classes=num_classes)
        raw_model = None  # <-- обязательно замените перед запуском
        self.model = NormalizedModel(raw_model, mean=mean, std=std)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, on_epoch=True)
        self.log("train_acc", acc, on_epoch=True)
        self.log("lr", self.trainer.optimizers[0].param_groups[0]["lr"], on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, on_epoch=True, sync_dist=True)
        self.log("val_acc", acc, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.max_epochs
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"},
        }


# TODO: гиперпараметры под вашу модель и датасет
model = MyClassifier(num_classes=10, lr=1e-3, weight_decay=0.05, max_epochs=100)

# Smoke-тест

Прогоняет один батч через весь пайплайн (данные → модель → шаг обучения → шаг валидации) и падает с понятной ошибкой, если что-то не сходится — до того, как тратить время на полное обучение.

In [ ]:
smoke_trainer = L.Trainer(
    fast_dev_run=True, accelerator="auto", devices=1,
    logger=False, enable_checkpointing=False,
)
smoke_trainer.fit(model, datamodule=dm)
print("Smoke-тест пройден")

# Обучение

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc", mode="max", save_top_k=1, filename="best"
)

trainer = L.Trainer(
    devices=1,
    max_epochs=model.hparams.max_epochs,
    accelerator="auto",
    precision="16-mixed",   # уберите, если модель чувствительна к точности (например, GAN)
    callbacks=[checkpoint_callback],
)
trainer.fit(model, datamodule=dm)

# Сохранение весов

In [ ]:
best_model = MyClassifier.load_from_checkpoint(checkpoint_callback.best_model_path)

# .model, а не весь LightningModule целиком — иначе в файле останется лишний
# префикс "model." перед каждым ключом (см. разбор этой ошибки в проекте раньше)
torch.save(best_model.model.cpu().state_dict(), 'my_model_weights.pth')